#### Final Scraping with enhanced code

In [ ]:
%pip install curl_cffi beautifulsoup4 mysql-connector-python numpy pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 3.4 MB/s  0:00:03m0:00:010:01


In [2]:
from curl_cffi import requests as cffi_requests
import bs4
import mysql.connector
import numpy

print("All installed correctly")

All installed correctly


In [3]:
# import all the required libraries
from curl_cffi import requests  # Drop-in replacement that impersonates Chrome TLS
from bs4 import BeautifulSoup
import re
import mysql.connector
from datetime import date
import numpy as np
import time
import random

In [4]:
# Categories that i want extract the data
categories = {#'Fashion' : ['Shirt', 'T-shirt', 'Jeans', 'Kurta', 'Sunglasses', 'Sandals', 'Heels', 'Slippers', 'Casual Shoes', 'Rucksacks', 'Formal Shoes', 'Jackets', 'Trolley', 'Jewellery'],
              'Mobile' : ['Iphone', 'Motorola', 'Vivo', 'Poco', 'Ai+', 'Redmi', 'Samsung', 'Realme', 'Oppo', 'Nothing', 'Google Pixel', 'Infinix', 'Tecno', 'Itel'],
              #'Beauty' : ['Skincare', 'Hair Care', 'Body Care', 'Bath & Spa', 'Womens Hygiene', 'Mens Grooming','Oral Care', 'winter picks', 'Luxe beauty', 'Perfumes', 'face wash', 'Roll-ons' ,'Eye Care', 'Lips', 'Deo', 'Derma Storre'],
              #'Electronics' : ['Audio', 'wearables', 'Grooming', '2 wheelers', 'camera', 'storage', 'Gaming', 'Healthcare', 'Laptop', 'tablet', 'pc accessiores', 'Mobile Cases', 'Chargers','Power Bank', 'small home devices', 'Gaming Laptop'],
              #'Home' : ['Cookware', 'utilities', 'Dinner ware', 'Decor', 'Furnishing', 'bath linen', 'Covers', 'bathroom', 'Cleaning', 'Wall decor', 'bed sheets', 'Blankets', 'Mattresses', 'sofas', 'Hardware', 'Lighting', 'poooja needs'],
              #'Applications' : ['Televisions','ACs','Laundry', 'Microwave oven', 'Refrigerators', 'Fans' ,'Heating gidgers', 'Inverters', 'Kitchen'], 
              #'Toy,Baby' : ['Baby toys', 'Toys & games', 'Stem toys', 'stationary', 'Diappers', 'wipes', 'pet toys'],
              #'Food' : ['Dry Fruits', 'Tea & Coffee', 'oil & ghee', 'choclates', 'Breakfast Essentials', 'pet food', 'proteins', 'Vitamin Supplement', 'Winter care', 'Condoms', 'Medical Supplies', 'Adult Diapers'],
              #'Auto Accessiories' : ['Lights', 'Tyre inflator', 'Batteries', 'Car mats', 'Subwoofers', 'Healmets', 'Engine Oils', 'Car Washer', 'Tyre', 'Riding gear'],
              #'Sports' : ['Badminton', 'Cycle', 'Balance Bikes', 'Exercise Bikes handpicked', 'yoga', 'camping', 'kids cycle', 'Treadmils', 'Gym Combo', 'Cricket', 'Ball Sports', 'Indoor Sports'],
              #'Books' : ['Guitars','Microphones', 'Keyboards', 'cajons', 'Amplifiers', 'Children Books', 'Boxsets', 'Fiction books', 'Non Fiction books', 'school books', 'Comics'],
              #'Furniture' : ['Mattresses', 'Office Chairs', 'Beds', 'ward drobes', 'office tables', 'kids furniture', 'sofa beds', 'Laptop tables', 'shoe racks', 'cofffe tables', 'Dining sets']
            }

In [5]:
def get_next_page(soup):
    next_page = soup.find('a', class_ = 's-pagination-next')
    if next_page and 'href' in next_page.attrs:
        return "https://www.amazon.in" + next_page['href']
    return None

In [ ]:
if __name__ == '__main__':

    def get_headers():
        return {
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
            'Accept-Language': 'en-IN,en;q=0.9',
            'Accept-Encoding': 'gzip, deflate, br',
            'Connection': 'keep-alive',
            'Upgrade-Insecure-Requests': '1',
            'Sec-Fetch-Dest': 'document',
            'Sec-Fetch-Mode': 'navigate',
            'Sec-Fetch-Site': 'none',
            'Sec-Fetch-User': '?1',
            'DNT': '1',
        }

    def is_blocked(html):
        return any(x in html.lower() for x in ["captcha", "robot check", "enter the characters", "type the characters"])

    today = date.today()

    # MySQL Connection
    conn = mysql.connector.connect(
        host="localhost",
        user="root",
        password="Enter your password",
    )
    cursor = conn.cursor()
    cursor.execute("CREATE DATABASE IF NOT EXISTS ecommerce")
    cursor.execute("USE ecommerce")
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS amazon_products (
            id INT AUTO_INCREMENT PRIMARY KEY,
            ASIN VARCHAR(20),
            title TEXT,
            price VARCHAR(50),
            rating VARCHAR(10),
            link TEXT,
            scrape_date DATE
        )
    """)

    insert_query = """
        INSERT INTO amazon_products
        (ASIN, title, price, rating, link, scrape_date)
        VALUES (%s, %s, %s, %s, %s, %s)
    """

    # KEY CHANGE: impersonate="chrome120" spoofs Chrome's TLS fingerprint
    session = requests.Session(impersonate="chrome120")
    session.headers.update(get_headers())

    # Warm-up: visit homepage first to get cookies like a real browser
    print("Warming up session...")
    session.get("https://www.amazon.in", timeout=15)
    time.sleep(random.uniform(8, 14))

    seen_asins = set()
    captcha_streak = 0  # track consecutive blocks

    for category, products in categories.items():
        print(f"\n===== CATEGORY: {category} =====")

        for product in products:
            print(f"\nSearching product: {product}")
            keyword = re.sub(r'\s+', '+', product)
            url = f"https://www.amazon.in/s?k={keyword}"
            page_no = 1

            while url and page_no <= 2:
                print(f"  Scraping page {page_no}")
                delay = random.uniform(8, 15)

                # Exponential backoff if blocked repeatedly
                if captcha_streak > 0:
                    backoff = min(60, delay * (2 ** captcha_streak))
                    print(f"  ⏳ Backoff: waiting {backoff:.1f}s after {captcha_streak} block(s)...")
                    time.sleep(backoff)
                else:
                    time.sleep(delay)

                try:
                    response = session.get(url, timeout=20, headers=get_headers())
                except Exception as e:
                    print(f"  ❌ Request error: {e}")
                    break

                if response.status_code != 200 or is_blocked(response.text):
                    captcha_streak += 1
                    print(f"⚠ CAPTCHA detected (streak: {captcha_streak}) → skipping keyword")
                    # Re-warm the session with a fresh visit
                    try:
                        session.get("https://www.amazon.in", timeout=15)
                    except:
                        pass
                    break

                captcha_streak = 0  # reset on success
                soup = BeautifulSoup(response.content, "html.parser")

                count = 0
                for item in soup.select("div.s-result-item[data-asin]"):
                    asin = item.get("data-asin")
                    if not asin or asin in seen_asins:
                        continue
                    seen_asins.add(asin)

                    title_el = item.select_one("h2 span")
                    price_el = item.select_one("span.a-price span.a-offscreen")
                    rating_el = item.select_one("span.a-icon-alt")

                    title = title_el.text.strip() if title_el else None
                    price = price_el.text.strip() if price_el else None
                    rating = rating_el.text.split()[0] if rating_el else None
                    link = "https://www.amazon.in/dp/" + asin

                    if not title or not link:
                        continue

                    cursor.execute(insert_query, (asin, title, price, rating, link, today))
                    count += 1

                conn.commit()
                print(f"  ✅ Inserted {count} products")

                next_btn = soup.select_one("a.s-pagination-next")
                url = "https://www.amazon.in" + next_btn["href"] if next_btn else None
                page_no += 1

    cursor.close()
    conn.close()
    print("\n✅ All categories & products scraped successfully")


Warming up session...

===== CATEGORY: Mobile =====

Searching product: Iphone
  Scraping page 1
  ✅ Inserted 18 products
  Scraping page 2
  ✅ Inserted 17 products

Searching product: Motorola
  Scraping page 1
  ✅ Inserted 20 products
  Scraping page 2
  ✅ Inserted 18 products

Searching product: Vivo
  Scraping page 1
  ✅ Inserted 19 products
  Scraping page 2
  ✅ Inserted 20 products

Searching product: Poco
  Scraping page 1
  ✅ Inserted 14 products
  Scraping page 2
  ✅ Inserted 16 products

Searching product: Ai+
  Scraping page 1
  ✅ Inserted 16 products
  Scraping page 2
  ✅ Inserted 16 products

Searching product: Redmi
  Scraping page 1
  ✅ Inserted 14 products
  Scraping page 2
  ✅ Inserted 15 products

Searching product: Samsung
  Scraping page 1
  ✅ Inserted 17 products
  Scraping page 2
  ✅ Inserted 19 products

Searching product: Realme
  Scraping page 1
  ✅ Inserted 18 products
  Scraping page 2
  ✅ Inserted 19 products

Searching product: Oppo
  Scraping page 1
  ✅ In

In [ ]:
random.uniform(6,10)

8.661481811790274